# ECCE 2027 Smoke Test (Kaggle): Dhaka Fold 0 (E1.0)
### 3-Epoch Verification, VRAM / Latency Profiling, & COCO JSON Export

Pivoted here from Colab because free-tier Colab GPU sessions were being preempted
mid-run. Kaggle's fixed weekly GPU quota tends to be far more stable for sustained
training.

**Attach two datasets before running** (Add Input, right sidebar):
1. `badodd-zenodo` — the extracted Zenodo BadODD archive (~10,032 images).
2. `badodd-ecce2027-overlay` — the 3.7 MB overlay (labels/splits/configs/coco_gt).

Settings: Accelerator = GPU T4 x1 (or P100), Internet = ON.

**Purpose:**
1. Locate and link the BadODD images + overlay, **failing hard** (not warning) on
   any missing image referenced by a split manifest.
2. Label integrity audit — proves training data isn't silently all-background.
3. Train **E1.0** for 3 epochs with hyperparameters pinned in `hyp.yaml`, on a
   version-pinned `ultralytics` install (cross-checked against the pin recorded
   in `hyp.yaml` at build time).
4. Export COCO-format predictions at `conf=0.001` on the 1,121 Chattogram pooled
   eval images.
5. Run `pycocotools` against the ground-truth JSON in a single evaluation pass and
   assert AP50 > 0 for `person`/`car` — proves category ids and box coordinates
   actually line up before committing to 8 full 100-epoch runs.

> Designed for **Kaggle 'Save & Run All'**. Zero manual input required.

In [ ]:
# ==============================================================================
# 1. HARDWARE & ENVIRONMENT VERIFICATION
# ==============================================================================
import os, sys, time, glob, json, shutil, subprocess

# Kaggle's GPU tier here is T4 x2 (no single-T4 option). Mask the second GPU so
# PyTorch/Ultralytics only ever see one clean device -- avoids any code path that
# branches on torch.cuda.device_count() > 1 and quietly reserves extra memory.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from pathlib import Path
import torch

PINNED_ULTRALYTICS = "8.4.155"

print('Python version:', sys.version)
print('PyTorch version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
print('CUDA device count (should be 1 after masking):', torch.cuda.device_count())

subprocess.run("nvidia-smi --query-gpu=index,name,memory.total,memory.used,memory.free "
               "--format=csv", shell=True)

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name} ({total_vram:.2f} GB VRAM)')
    torch.cuda.reset_peak_memory_stats(0)
else:
    print('WARNING: No GPU detected! Enable GPU accelerator in notebook settings.')

subprocess.run(f"pip install -q ultralytics=={PINNED_ULTRALYTICS} pycocotools", shell=True, check=True)
import ultralytics
assert ultralytics.__version__ == PINNED_ULTRALYTICS, (
    f"Ultralytics version drift: installed {ultralytics.__version__}, expected {PINNED_ULTRALYTICS}"
)
print('Ultralytics version:', ultralytics.__version__)

from ultralytics import YOLO

In [ ]:
# ==============================================================================
# 2. DATASET & OVERLAY DISCOVERY -- FAIL HARD on any missing image
# ==============================================================================
print('=== Scanning /kaggle/input for Dataset & Overlay ===')

# If badodd.zip is attached unextracted (kept as one file to avoid Kaggle's dataset
# upload pipeline choking on ~20,000 individual image+label files), extract it once
# to local disk before scanning for images.
badodd_zips = glob.glob('/kaggle/input/**/badodd.zip', recursive=True)
if badodd_zips and not glob.glob('/kaggle/input/**/*.jpg', recursive=True):
    print(f'Found badodd.zip at {badodd_zips[0]}. Extracting to /kaggle/working/badodd_images...')
    os.makedirs('/kaggle/working/badodd_images', exist_ok=True)
    subprocess.run(f"unzip -q {badodd_zips[0]} -d /kaggle/working/badodd_images", shell=True, check=True)
    IMAGE_SEARCH_ROOT = '/kaggle/working/badodd_images'
else:
    IMAGE_SEARCH_ROOT = '/kaggle/input'

all_input_imgs = [p for p in glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.jpg', recursive=True) +
                        glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.png', recursive=True)
                   if not os.path.basename(p).startswith('._')]
img_lookup = {os.path.basename(p): p for p in all_input_imgs}
print(f'Indexed {len(img_lookup)} images from {IMAGE_SEARCH_ROOT}.')
assert len(img_lookup) >= 10000, (
    f"Expected ~10,032 BadODD images, found only {len(img_lookup)} -- "
    f"is the badodd-zenodo(-bundle) dataset attached (not the truncated Kaggle competition copy)?"
)

overlay_roots = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'splits' in dirs and 'labels' in dirs:
        overlay_roots.append(root)

overlay_zips = glob.glob('/kaggle/input/**/badodd_ecce2027_overlay*.zip', recursive=True)
overlay_src = None
if overlay_roots:
    overlay_src = overlay_roots[0]
    print(f'Found overlay directory at: {overlay_src}')
elif overlay_zips:
    print(f'Found overlay zip at: {overlay_zips[0]}. Unzipping to /kaggle/working/overlay...')
    subprocess.run(f"unzip -q {overlay_zips[0]} -d /kaggle/working/overlay", shell=True, check=True)
    overlay_src = '/kaggle/working/overlay'
else:
    raise FileNotFoundError(
        'Could not locate the ECCE 2027 overlay package in /kaggle/input! '
        'Attach the badodd-ecce2027-overlay(-bundle) dataset.'
    )

assert os.path.isdir(os.path.join(overlay_src, 'labels'))
assert os.path.isdir(os.path.join(overlay_src, 'splits'))
assert os.path.isdir(os.path.join(overlay_src, 'coco_gt'))
assert os.path.isdir(os.path.join(overlay_src, 'configs'))

WORK_DATA = '/kaggle/working/data'
WORK_SPLITS = os.path.join(WORK_DATA, 'splits')
WORK_LABELS = os.path.join(WORK_DATA, 'labels')
WORK_IMAGES = os.path.join(WORK_DATA, 'images')
os.makedirs(WORK_SPLITS, exist_ok=True)
os.makedirs(WORK_LABELS, exist_ok=True)
os.makedirs(WORK_IMAGES, exist_ok=True)

label_files = glob.glob(os.path.join(overlay_src, 'labels', '*.txt'))
for lf in label_files:
    dest = os.path.join(WORK_LABELS, os.path.basename(lf))
    if not os.path.exists(dest):
        try:
            os.symlink(lf, dest)
        except OSError:
            shutil.copy(lf, dest)
print(f'Linked {len(label_files)} label files.')

for bname, src_p in img_lookup.items():
    dest = os.path.join(WORK_IMAGES, bname)
    if not os.path.exists(dest):
        try:
            os.symlink(src_p, dest)
        except OSError:
            shutil.copy(src_p, dest)
print(f'Linked {len(img_lookup)} image files.')

# Resolve every split manifest to absolute WORK_IMAGES paths; FAIL HARD on any miss
manifest_files = glob.glob(os.path.join(overlay_src, 'splits', '*.txt'))
resolved_counts = {}
for mf in manifest_files:
    mname = os.path.basename(mf)
    with open(mf, 'r') as f:
        bases = [os.path.basename(l.strip()) for l in f if l.strip()]
    resolved = []
    missing_here = []
    for b in bases:
        p = os.path.join(WORK_IMAGES, b)
        if os.path.exists(p):
            resolved.append(p)
        else:
            missing_here.append(b)
    if missing_here:
        raise FileNotFoundError(
            f"{mname}: {len(missing_here)} images referenced in the manifest are missing, "
            f"e.g. {missing_here[:5]}. Do not proceed -- check the attached dataset."
        )
    with open(os.path.join(WORK_SPLITS, mname), 'w') as f:
        f.writelines(p + '\n' for p in resolved)
    resolved_counts[mname] = len(resolved)

print(f'Resolved {len(manifest_files)} manifests, ALL images present. Sample counts:')
for k in ('dhaka_fold0_train_matched.txt', 'dhaka_fold0_train_matched_monitor.txt', 'ctg_pooled_eval.txt'):
    print(f'  {k}: {resolved_counts.get(k)}')

print('Dataset & Overlay setup complete!')

In [ ]:
# ==============================================================================
# 3. WRITE YOLO DATASET CONFIG & LOAD PINNED HYPERPARAMETERS
# ==============================================================================
import yaml

with open(os.path.join(overlay_src, 'configs', 'hyp.yaml')) as f:
    hyp = yaml.safe_load(f)
assert hyp['ultralytics_version'] == PINNED_ULTRALYTICS, (
    f"hyp.yaml pins ultralytics=={hyp['ultralytics_version']} but this runtime has {PINNED_ULTRALYTICS}"
)
print('Loaded pinned hyperparameters from hyp.yaml:')
print(hyp)

CLASS_NAMES = {
    0: 'auto_rickshaw', 1: 'bicycle', 2: 'bus', 3: 'car', 4: 'cart_vehicle',
    5: 'construction_vehicle', 6: 'motorbike', 7: 'person', 8: 'priority_vehicle',
    9: 'three_wheeler', 10: 'truck',
}

smoke_yaml = {
    'path': WORK_DATA,
    'train': os.path.join(WORK_SPLITS, 'dhaka_fold0_train_matched.txt'),
    'val': os.path.join(WORK_SPLITS, 'dhaka_fold0_train_matched_monitor.txt'),
    'names': CLASS_NAMES,
}

SMOKE_YAML_PATH = '/kaggle/working/data_smoke.yaml'
with open(SMOKE_YAML_PATH, 'w') as f:
    yaml.dump(smoke_yaml, f, sort_keys=False)

print(f'\nWrote dataset configuration to: {SMOKE_YAML_PATH}')
print('Train manifest:', smoke_yaml['train'])
print('Val monitoring manifest:', smoke_yaml['val'])

In [ ]:
# ==============================================================================
# 3b. LABEL INTEGRITY AUDIT -- run BEFORE training. Catches silent background-only runs.
# ==============================================================================
train_manifest = smoke_yaml['train']
with open(train_manifest) as f:
    train_imgs = [l.strip() for l in f if l.strip()]

total_boxes_gt = 0
n_background = 0
for img_p in train_imgs:
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(WORK_LABELS, stem + '.txt')
    if not os.path.exists(lbl_p) or os.path.getsize(lbl_p) == 0:
        n_background += 1
        continue
    with open(lbl_p) as f:
        n = sum(1 for line in f if line.strip())
    total_boxes_gt += n
    if n == 0:
        n_background += 1

bg_rate = n_background / len(train_imgs)
print(f'Train Images in Manifest: {len(train_imgs)} | Total Ground Truth Boxes: {total_boxes_gt} | '
      f'Background Images: {n_background} ({bg_rate*100:.2f}%)')
assert total_boxes_gt > 0, 'FAIL: zero ground-truth boxes found -- labels did not link correctly.'
assert bg_rate < 0.05, f'FAIL: background image rate {bg_rate*100:.2f}% >= 5% -- label linkage is broken.'
print('Label integrity check PASSED.')

In [ ]:
# ==============================================================================
# 4. RUN SMOKE TRAINING (E1.0 -- 3 EPOCHS)
# ==============================================================================
PROJECT_DIR = '/kaggle/working/runs/train'
EXP_NAME = 'e1_0_smoke'
EXP_DIR = os.path.join(PROJECT_DIR, EXP_NAME)
LAST_CKPT = os.path.join(EXP_DIR, 'weights', 'last.pt')

can_resume = os.path.exists(LAST_CKPT)
model = YOLO(LAST_CKPT if can_resume else hyp['model'])
print('Resuming from checkpoint.' if can_resume else f"Starting fresh from {hyp['model']}.")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(0)

SMOKE_EPOCHS = 3
t0_train = time.time()
train_results = model.train(
    data=SMOKE_YAML_PATH,
    epochs=SMOKE_EPOCHS,
    patience=hyp['patience'],
    batch=hyp['batch'],
    imgsz=hyp['imgsz'],
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    optimizer=hyp['optimizer'],
    lr0=hyp['lr0'], lrf=hyp['lrf'], momentum=hyp['momentum'], weight_decay=hyp['weight_decay'],
    warmup_epochs=hyp['warmup_epochs'], warmup_momentum=hyp['warmup_momentum'], warmup_bias_lr=hyp['warmup_bias_lr'],
    box=hyp['box'], cls=hyp['cls'], dfl=hyp['dfl'],
    hsv_h=hyp['hsv_h'], hsv_s=hyp['hsv_s'], hsv_v=hyp['hsv_v'],
    degrees=hyp['degrees'], translate=hyp['translate'], scale=hyp['scale'], shear=hyp['shear'],
    perspective=hyp['perspective'], flipud=hyp['flipud'], fliplr=hyp['fliplr'],
    mosaic=hyp['mosaic'], mixup=hyp['mixup'], copy_paste=hyp['copy_paste'],
    seed=hyp['seed'], deterministic=hyp['deterministic'],
    project=PROJECT_DIR, name=EXP_NAME, exist_ok=True, save=True, resume=can_resume, verbose=True,
)

total_train_time = time.time() - t0_train
time_per_epoch = total_train_time / SMOKE_EPOCHS
projected_100_epochs_hours = (time_per_epoch * 100.0) / 3600.0

peak_vram_gb = 0.0
if torch.cuda.is_available():
    peak_vram_gb = torch.cuda.max_memory_allocated(0) / (1024**3)

assert os.path.exists(LAST_CKPT), f'ERROR: {LAST_CKPT} was not created!'
print('\n' + '='*60)
print('           SMOKE TRAINING BENCHMARK RESULTS           ')
print('='*60)
print(f'Total Smoke Training Time (3 Epochs): {total_train_time:.2f} s')
print(f'Measured Duration Per Epoch:          {time_per_epoch:.2f} s')
print(f'Projected Full 100-Epoch Duration:    {projected_100_epochs_hours:.2f} hours')
print(f'Peak GPU VRAM Allocated:              {peak_vram_gb:.2f} GB')
print(f'Checkpoint Generated:                 {LAST_CKPT}')
print('='*60)

run_info = {
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'total_vram_gb': torch.cuda.get_device_properties(0).total_memory/(1024**3) if torch.cuda.is_available() else None,
    'torch_version': torch.__version__,
    'ultralytics_version': PINNED_ULTRALYTICS,
    'seconds_per_epoch': time_per_epoch,
    'projected_100_epoch_hours': projected_100_epochs_hours,
    'peak_vram_gb': peak_vram_gb,
}
os.makedirs('/kaggle/working/results/logs', exist_ok=True)
with open('/kaggle/working/results/logs/e1_0_smoke_run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)
print(json.dumps(run_info, indent=2))

In [ ]:
# ==============================================================================
# 5. EXPORT COCO-JSON PREDICTIONS AT conf=0.001 (CRITICAL)
# ==============================================================================
import gc

if torch.cuda.is_available():
    print(f'GPU memory before cleanup: allocated={torch.cuda.memory_allocated(0)/1e9:.2f} GB, '
          f'reserved={torch.cuda.memory_reserved(0)/1e9:.2f} GB')

# Ultralytics' trainer keeps references (EMA model, optimizer state, validator) that
# `del model` alone does not release -- explicitly tear those down first.
for name in ('trainer', 'validator', 'ema'):
    try:
        obj = getattr(model, name, None)
        if obj is not None:
            delattr(model, name)
            del obj
    except Exception:
        pass
try:
    del model
except NameError:
    pass
try:
    del train_results
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    allocated_after = torch.cuda.memory_allocated(0) / 1e9
    reserved_after = torch.cuda.memory_reserved(0) / 1e9
    print(f'GPU memory after cleanup:  allocated={allocated_after:.2f} GB, reserved={reserved_after:.2f} GB')
    assert allocated_after < 1.0, (
        f'FAIL: {allocated_after:.2f} GB still allocated after cleanup -- training left memory '
        f'resident that a fresh model + inference batch will not fit alongside. Do not proceed; '
        f'this needs a real fix (e.g. splitting train and eval into separate notebook runs), '
        f'not a smaller batch size.'
    )

EVAL_MANIFEST = os.path.join(WORK_SPLITS, 'ctg_pooled_eval.txt')
RESULTS_DIR = '/kaggle/working/results/predictions'
os.makedirs(RESULTS_DIR, exist_ok=True)
OUTPUT_JSON = os.path.join(RESULTS_DIR, 'e1_0_smoke_ctg_pooled_eval_preds_conf0001.json')

with open(EVAL_MANIFEST, 'r') as f:
    eval_images = [l.strip() for l in f if l.strip()]

print(f'Evaluating last.pt model on {len(eval_images)} Chattogram evaluation images...')
eval_model = YOLO(LAST_CKPT)

t0_infer = time.time()
results_gen = eval_model.predict(
    source=eval_images,
    conf=0.001,
    iou=0.7,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    batch=8,  # reduced from 16 for extra safety margin on a shared 15 GB T4
    stream=True,
    verbose=False,
    workers=0,  # avoid multi-worker dataloader instability
)

coco_predictions = []
class_counts = {cid: 0 for cid in CLASS_NAMES}
total_boxes = 0

for idx, r in enumerate(results_gen):
    img_stem = Path(r.path).stem
    boxes = r.boxes
    if boxes is None or len(boxes) == 0:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()
    clss = boxes.cls.cpu().numpy().astype(int)

    for i in range(len(boxes)):
        x1, y1, x2, y2 = xyxy[i]
        w = x2 - x1
        h = y2 - y1
        cid = int(clss[i])
        coco_predictions.append({
            'image_id': img_stem,
            'category_id': cid,
            'bbox': [round(float(x1), 2), round(float(y1), 2), round(float(w), 2), round(float(h), 2)],
            'score': round(float(confs[i]), 5),
        })
        total_boxes += 1
        if cid in class_counts:
            class_counts[cid] += 1

    if (idx + 1) % 200 == 0:
        print(f'  {idx + 1}/{len(eval_images)} images processed...')

total_infer_time = time.time() - t0_infer
fps = len(eval_images) / total_infer_time

print(f'\nInference finished in {total_infer_time:.2f} s ({fps:.1f} FPS)')
print(f'Total exported detections at conf>=0.001: {total_boxes}')

print('\nDetections by Class:')
for cid, cname in CLASS_NAMES.items():
    print(f'  [{cid:2d}] {cname:22s}: {class_counts[cid]}')

with open(OUTPUT_JSON, 'w') as f:
    json.dump(coco_predictions, f)

json_size_mb = os.path.getsize(OUTPUT_JSON) / (1024*1024)
print(f'\nExported JSON: {OUTPUT_JSON} ({json_size_mb:.2f} MB)')
assert total_boxes > 0, 'FAIL: zero predictions exported.'

In [ ]:
# ==============================================================================
# 5b. PYCOCOTOOLS ALIGNMENT CHECK -- proves category ids / bbox coords / image ids match GT
# ==============================================================================
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

GT_JSON = os.path.join(overlay_src, 'coco_gt', 'ctg_pooled_eval_coco_gt.json')
coco_gt = COCO(GT_JSON)
coco_dt = coco_gt.loadRes(OUTPUT_JSON)

ev = COCOeval(coco_gt, coco_dt, iouType='bbox')
ev.evaluate(); ev.accumulate(); ev.summarize()
mAP50_95 = ev.stats[0]
mAP50 = ev.stats[1]

print(f'\nOverall mAP50: {mAP50:.4f} | mAP50-95: {mAP50_95:.4f}')

print('\nPer-class AP50:')
cat_ids = ev.params.catIds
per_class_ap50 = {}
for k, cid in enumerate(cat_ids):
    ap50_arr = ev.eval['precision'][0, :, k, 0, -1]
    ap50 = ap50_arr[ap50_arr > -1].mean() if (ap50_arr > -1).any() else float('nan')
    cname = CLASS_NAMES[cid]
    per_class_ap50[cname] = ap50
    print(f'  [{cid:2d}] {cname:22s}: AP50={ap50:.4f}')

assert mAP50 >= 0
assert per_class_ap50.get('person', 0) > 0 or per_class_ap50.get('car', 0) > 0, (
    'FAIL: AP50 is 0 for both person and car after 3 epochs -- category ids or bbox coords '
    'are almost certainly misaligned between predictions and ground truth.'
)
print('\nAlignment check PASSED: predictions and ground truth agree on ids/coordinates.')

In [ ]:
# ==============================================================================
# 6. OUTPUT VERIFICATION & SMOKE TEST CHECKLIST
# ==============================================================================
print('='*70)
print('                     SMOKE TEST VERIFICATION CHECKLIST                ')
print('='*70)

assert os.path.exists(LAST_CKPT), f'FAIL: {LAST_CKPT} missing!'
ckpt_size_mb = os.path.getsize(LAST_CKPT) / (1024*1024)
print(f'1. [PASS] Checkpoint last.pt exists ({ckpt_size_mb:.2f} MB)')

results_csv = os.path.join(EXP_DIR, 'results.csv')
assert os.path.exists(results_csv), f'FAIL: {results_csv} missing!'
shutil.copy(results_csv, '/kaggle/working/results/logs/e1_0_smoke_results.csv')
shutil.copy(os.path.join(EXP_DIR, 'args.yaml'), '/kaggle/working/results/logs/e1_0_smoke_args.yaml')
print(f'2. [PASS] Training metrics results.csv exists')

assert os.path.exists(OUTPUT_JSON), f'FAIL: {OUTPUT_JSON} missing!'
with open(OUTPUT_JSON, 'r') as f:
    verified_preds = json.load(f)
assert len(verified_preds) == total_boxes, 'FAIL: JSON prediction record count mismatch!'
assert len(verified_preds) > 0, 'FAIL: Zero predictions generated!'
print(f'3. [PASS] COCO Predictions JSON valid ({len(verified_preds)} bounding boxes)')

print(f'4. [PASS] Training throughput: {time_per_epoch:.2f} s/epoch')
print(f'   -> Projected 100 epochs will take ~{projected_100_epochs_hours:.2f} hours (well within Kaggle 12-hour limit)')
print(f'5. [PASS] GPU VRAM peak: {peak_vram_gb:.2f} GB')
print(f'6. [PASS] pycocotools alignment check: mAP50={mAP50:.4f}, '
      f"person AP50={per_class_ap50.get('person', float('nan')):.4f}, "
      f"car AP50={per_class_ap50.get('car', float('nan')):.4f}")

weights_dir = '/kaggle/working/results/weights'
os.makedirs(weights_dir, exist_ok=True)
shutil.copy(LAST_CKPT, os.path.join(weights_dir, 'e1_0_smoke_last.pt'))

print('='*70)
print('>>> ALL SMOKE TEST CHECKS PASSED SUCCESSFULLY! <<<')
print('Outputs are under /kaggle/working/results/ -- click Save Version to')
print('persist them, then download logs/, weights/, predictions/ to your laptop.')
print('You are cleared to run full 100-epoch training.')
print('='*70)